# Decisions & allocation

Four questions the return metrics cannot answer.

1. Did picking stocks beat indexing the money as it arrived?
2. Which sales cost the most?
3. How many bets is this really?
4. Where is the risk, as opposed to the money?

> **Clear outputs before committing.** These tables hold real position sizes.
> This repository is public. `nbstripout` handles it.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data.investing_csv import build_portfolio
from data.market_data import MarketDataFetcher
from analytics.timeseries import PortfolioTimeSeries
from analytics.decisions import (cashflow_matched_benchmark, sale_opportunity_cost,
                                 realisation_asymmetry)
from analytics.allocation import (covariance, correlation, concentration_bets,
                                  effective_bets, risk_contributions, segment_breakdown,
                                  min_variance_weights, risk_parity_weights,
                                  compare_allocations, efficient_frontier,
                                  portfolio_volatility, asset_history)

pd.options.display.float_format = '{:,.3f}'.format

CSV_PATH  = os.environ.get('HOLDINGS_CSV', os.path.join('..', 'private', 'holdings.csv'))
BASE      = 'USD'
BENCHMARK = 'SPY'
MAX_WEIGHT = 0.25          # cap on any single holding in the optimised mixes
LOOKBACK_DAYS = 730        # history used for covariance, independent of purchase dates

fetcher = MarketDataFetcher()
portfolio, report = build_portfolio(CSV_PATH, name='imported', base_currency=BASE)
ts = PortfolioTimeSeries(portfolio, fetcher=fetcher)

held = [p.ticker for p in portfolio.open_positions()]

# A common window, not the intersection of each holding's ownership period.
# Intersecting those leaves only weeks once a recent purchase is included,
# which is not enough history to estimate a covariance from.
asset_returns = asset_history(held, fetcher, base_currency=BASE,
                             currencies=ts.instrument_currencies,
                             lookback_days=LOOKBACK_DAYS)
cov = covariance(asset_returns)
weights = ts.current_weights()[held]
weights = weights / weights.sum()

print(f'{len(held)} holdings, {len(asset_returns)} days of history')
if len(asset_returns) < 10 * len(held):
    print(f'  WARNING: {len(asset_returns)} observations for {len(held)} assets is thin.')
    print(f'  The covariance is poorly conditioned; treat section 6 as indicative only.')

## 1 · Did the stock picking pay?

Total return against an index is flattering or punishing purely because of when
the money arrived. This buys the benchmark with each contribution on the day it
landed — the same money, the same dates.

Sales that funded purchases are correctly absent: they were never new money, so
the benchmark is not credited with them.

In [ ]:
matched = cashflow_matched_benchmark(ts, BENCHMARK, fetcher=fetcher)

print(f"contributed          {matched['contributed']:>12,.2f} {BASE}")
print(f"your portfolio       {matched['portfolio_value']:>12,.2f}   {matched['portfolio_gain']:+.1%}")
print(f"{BENCHMARK}, same deposits  {matched['benchmark_value']:>12,.2f}   {matched['benchmark_gain']:+.1%}")
print(f"{'-'*46}")
print(f"active management    {matched['active_value_added']:>+12,.2f}   "
      f"{matched['active_value_added']/matched['contributed']:+.1%} of capital")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(matched['portfolio_series'].index, matched['portfolio_series'].values,
        linewidth=1.5, label='Your portfolio', color='steelblue')
ax.plot(matched['benchmark_series'].index, matched['benchmark_series'].values,
        linewidth=1.5, label=f'{BENCHMARK}, same deposits', color='grey', linestyle='--')
ax.set_title(f'Same money, same dates — you vs {BENCHMARK}')
ax.set_ylabel(BASE)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 2 · What did the sales cost?

Proceeds against what the holding would be worth today, converted at the rate on
each date. Positive is money given up.

Read it as the cost of each sale **in isolation** — it assumes the proceeds sat
idle, and they did not. Section 1 is the bottom line that accounts for
redeployment. This is for spotting which specific holdings were worth keeping.

In [ ]:
sales = sale_opportunity_cost(portfolio, fetcher, ts)
sales[['ticker', 'currency', 'exit_date', 'proceeds', 'worth_now', 'sale_cost', 'sale_cost_pct']]

In [ ]:
top = sales.head(10).iloc[::-1]
colors = ['indianred' if x > 0 else 'seagreen' for x in top['sale_cost']]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(top['ticker'], top['sale_cost'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Cost of each sale (red = sold too early)')
ax.set_xlabel(BASE)
ax.grid(alpha=0.3, axis='x')
plt.show()

print(f"net across every sale: {sales['sale_cost'].sum():+,.2f} {BASE}")

### Are gains realised while losses are held?

A high share of closes at a gain sitting next to open positions at a loss is the
disposition effect. Worth seeing before the next sale, not a year later.

In [ ]:
a = realisation_asymmetry(portfolio, ts)

pd.Series({
    'closed positions':      a['closed_positions'],
    'closed at a gain':      a['closed_at_gain'],
    'win rate on closes':    a['closed_win_rate'],
    'avg realised gain':     a['avg_realised_gain'],
    'open positions':        a['open_positions'],
    'open at a loss':        a['open_at_loss'],
    'avg open loss':         a['avg_open_loss'],
}).to_frame('value')

## 3 · How many bets is this really?

**Looks like** counts positions and ignores how they move — the inverse
Herfindahl on weights. **Actually** is the squared diversification ratio, which
accounts for correlation. A wide gap means holdings that move together: several
names wearing one bet's clothing.

In [ ]:
looks_like = concentration_bets(weights)
actually   = effective_bets(weights, cov)

print(f"holdings                {len(held)}")
print(f"looks like              {looks_like:.1f} bets   (weights only)")
print(f"actually                {actually:.1f} bets   (correlation-aware)")
print(f"portfolio volatility    {portfolio_volatility(weights, cov):.1%}")

In [ ]:
# Correlation, ordered so clusters sit together.
corr = correlation(asset_returns)
order = corr.mean().sort_values(ascending=False).index
corr = corr.loc[order, order]

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr.to_numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr)), corr.index, fontsize=8)
ax.set_title('Correlation of daily returns (base currency)')
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

## 4 · Where is the risk?

Weight says where the money is. Risk contribution says where the volatility comes
from. A small, jumpy holding can easily be several times its weight — which means
the position you think is largest may not be the one driving your outcome.

In [ ]:
risk = risk_contributions(weights, cov)
side_by_side = pd.DataFrame({'weight': weights, 'risk share': risk})
side_by_side['risk / weight'] = side_by_side['risk share'] / side_by_side['weight']
side_by_side = side_by_side.sort_values('risk share', ascending=False)
side_by_side

In [ ]:
plot = side_by_side.iloc[::-1]
y = np.arange(len(plot))

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(y - 0.2, plot['weight'], height=0.4, label='share of money', color='steelblue')
ax.barh(y + 0.2, plot['risk share'], height=0.4, label='share of risk', color='indianred')
ax.set_yticks(y, plot.index, fontsize=9)
ax.set_title('Money vs risk')
ax.legend()
ax.grid(alpha=0.3, axis='x')
plt.show()

## 5 · Segments

Grouped by quote currency by default. For a part-Turkish book that is the split
that matters: those holdings answer to a different market and a different
currency, and averaging them with the US book hides both.

Pass any mapping to `segment_breakdown(ts, {'MSFT': 'quality', ...})` to cut it
another way — by theme, sector, or conviction.

In [ ]:
segment_breakdown(ts)

## 6 · Could the same holdings be arranged better?

Two of these need **no return forecast** and are the ones worth acting on:

- **min variance** — the quietest long-only mix of what you already hold
- **risk parity** — every holding contributing an equal share of risk

**max sharpe** and the frontier need expected returns, and a historical mean over
this short a window is mostly noise. Read `expected_return` below as a
description of what happened, not a forecast. The `volatility` and
`effective_bets` columns are the trustworthy ones.

In [ ]:
expected = asset_returns.mean() * 252          # weak input -- see the note above

min_var = min_variance_weights(cov, max_weight=MAX_WEIGHT)
parity  = risk_parity_weights(cov, max_weight=MAX_WEIGHT)

compare_allocations({
    'current':      weights,
    'min variance': min_var,
    'risk parity':  parity,
}, expected, cov)

In [ ]:
suggestions = pd.DataFrame({
    'current': weights,
    'min variance': min_var,
    'risk parity': parity,
}).sort_values('min variance', ascending=False)
suggestions['change to min var'] = suggestions['min variance'] - suggestions['current']
suggestions

In [ ]:
# Where the current portfolio sits against the frontier of its own holdings.
frontier = efficient_frontier(expected, cov, points=25, max_weight=MAX_WEIGHT)

fig, ax = plt.subplots(figsize=(9, 5))
if not frontier.empty:
    ax.plot(frontier['volatility'], frontier['expected_return'],
            linewidth=1.5, color='grey', label='efficient frontier')

for name, w, colour, marker in [
    ('current', weights, 'steelblue', 'o'),
    ('min variance', min_var, 'seagreen', 's'),
    ('risk parity', parity, 'darkorange', '^'),
]:
    ax.scatter(portfolio_volatility(w, cov), float(w @ expected.reindex(cov.index)),
               s=110, color=colour, marker=marker, zorder=3, label=name)

ax.set_xlabel('volatility (annualised)')
ax.set_ylabel('return (annualised, historical)')
ax.set_title('Same holdings, different weights')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### Reading this honestly

The frontier is drawn from historical means, so its *height* is unreliable. What
it does show usefully is horizontal distance: how much volatility the current mix
carries for the return it produced, and how far left the same holdings could sit.

`min variance` will always pile into the quietest names and hit the weight cap.
That is what it optimises for, not a claim that those names are better.